# Notebook 04: pHH3 mitotic cell density mapping

**Paper section:** Results — Spatial patterns of mitotic activity are preserved across global expansion  
**Paper figures:** Fig. 6

This notebook maps the spatial distribution of mitotic cells (pHH3⁺) onto the neural tube surface.

**Inputs:**
- Lumen surface mesh (`.ply`)
- 3D coordinates of pHH3⁺ cell spots (detected in Imaris, exported as CSV)

**Method:**
1. Filter spots to those inside the tissue boundary
2. Compute a local density field D(x) = number of spots within radius R = 100 µm of each voxel
3. Project the density field onto the mesh via inverse distance weighting (IDW) interpolation

In [ ]:
from vedo import settings
settings.default_backend = "vtk"

import spatchcocking as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load mesh and cell coordinates

In [ ]:
mesh = sp.get_mesh("../data/meshes/HH17/HH17_embryo1_lumen.ply")

# Load pHH3+ spot coordinates (exported from Imaris as CSV)
# Expected columns: 'Position X', 'Position Y', 'Position Z' (in µm)
spots_df = pd.read_csv("path/to/HH17_embryo1_pHH3_spots.csv")
spots = spots_df[["Position X", "Position Y", "Position Z"]].values
print(f"Loaded {len(spots)} pHH3+ cells")

## Compute density field and transfer to mesh

In [ ]:
# Transfer pHH3+ density onto the mesh via IDW interpolation
# R = 100 µm: chosen to span the full tissue thickness at both stages
mesh = sp.transfer_points_to_mesh(mesh, spots, radius=100.0)

density = mesh.pointdata["pHH3_density"]
print(f"Density: min={density.min():.1f}, max={density.max():.1f}, mean={density.mean():.1f} cells")

## Visualize on 3D mesh

In [ ]:
from vedo import Plotter

mesh.pointdata.select("pHH3_density")
mesh.cmap("hot_r", vmin=0, vmax=density.max())

p = Plotter(offscreen=True)
p.show(mesh, axes=0)
p.screenshot("pHH3_density_3d.png")
from IPython.display import Image
Image("pHH3_density_3d.png")

## Save mesh with density data

In [ ]:
sp.save_mesh(mesh, "../data/meshes/HH17/HH17_embryo1_lumen_pHH3.ply")